# Dutch North Sea Decommissioning — Exploratory Notebook

Seven sections, built and committed one at a time.

| Section | Purpose |
|---|---|
| 1 | Fetch raw borehole data from NLOG API |
| 2 | Inspect shape, dtypes, uniques, nulls |
| 3 | Filter offshore, split Group A / B / ghost |
| 4 | Operator analysis on Group A |
| 5 | Production crossref — informal cessation |
| 6 | Join and summarise |
| 7 | Sense check against data diary |

**Data dirs** (`netherlands/data/`) are gitignored. Raw JSON and all CSVs  
live only on your local machine unless explicitly exported.

---
## Section 1 — Fetch

Uses `requests.Session` to:
1. `GET` the datacenter overview page — sets the required session cookie.
2. `POST` (empty body) to the boreholes endpoint — returns all 6,723 wells.

Saves raw JSON to `netherlands/data/raw/nlog_boreholes_raw.json`.

In [ ]:
import json
import logging
from pathlib import Path

import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Paths — notebook lives in netherlands/notebooks/, data two levels over.
# Path.cwd() resolves to the notebook directory when launched normally.
# ---------------------------------------------------------------------------
NOTEBOOKS_DIR = Path.cwd()
NETHERLANDS_ROOT = NOTEBOOKS_DIR.parent          # netherlands/
DATA_RAW = NETHERLANDS_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED = NETHERLANDS_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = NETHERLANDS_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSON_PATH = DATA_RAW / "nlog_boreholes_raw.json"

print(f"Project root : {NETHERLANDS_ROOT}")
print(f"Raw JSON     : {RAW_JSON_PATH}")

In [ ]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_SEED_URL = "https://www.nlog.nl/datacenter/brh-overview"
_BOREHOLES_URL = "https://www.nlog.nl/nlog-mapviewer/rest/brh/boreholes"
_TIMEOUT = 60  # seconds
_EXPECTED_COUNT = 6_723
_EXPECTED_FIELDS = {
    "boreholeDbk",
    "boreholeName",
    "shortName",
    "clientOrgName",
    "legalOwnerName",
    "statusDescription",
    "resultCode",
    "onOffshore",
    "startDate",
    "endDate",
    "confidentialityDate",
    "blockCd",
}

In [ ]:
def seed_session() -> requests.Session:
    """Open a session and hit the datacenter overview page to set the required cookie."""
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; research/1.0)"})
    resp = session.get(_SEED_URL, timeout=_TIMEOUT)
    resp.raise_for_status()
    logger.info("Seed GET %s → %s (cookies: %s)", _SEED_URL, resp.status_code, list(session.cookies.keys()))
    return session


def fetch_boreholes(session: requests.Session) -> list:
    """POST to the boreholes endpoint and return the parsed JSON array."""
    resp = session.post(_BOREHOLES_URL, timeout=_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    logger.info("Boreholes POST → %s records", len(data))
    return data


def validate_record_count(data: list) -> None:
    """Warn loudly if the record count differs significantly from the expected total."""
    count = len(data)
    delta = abs(count - _EXPECTED_COUNT)
    if delta > 50:
        logger.warning(
            "COUNT MISMATCH — got %d records, expected ~%d (delta %d). "
            "Stop and flag before proceeding.",
            count, _EXPECTED_COUNT, delta,
        )
    else:
        logger.info("Record count %d — within 50 of expected %d ✓", count, _EXPECTED_COUNT)


def validate_field_names(record: dict) -> None:
    """Check that every expected field is present in the first record."""
    actual = set(record.keys())
    missing = _EXPECTED_FIELDS - actual
    extra = actual - _EXPECTED_FIELDS
    if missing:
        logger.warning("MISSING FIELDS: %s", missing)
    if extra:
        logger.info("Extra fields not in spec (note for data diary): %s", extra)
    if not missing:
        logger.info("All expected fields present ✓")


def save_raw(data: list, path: Path) -> None:
    """Write the raw JSON array to disk."""
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2))
    logger.info("Saved %d records to %s", len(data), path)

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 1
# ---------------------------------------------------------------------------

session = seed_session()
raw_data = fetch_boreholes(session)

validate_record_count(raw_data)
validate_field_names(raw_data[0])

print("\n--- First record ---")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))

save_raw(raw_data, RAW_JSON_PATH)

---
## Section 2 — Inspect

Load raw JSON into a pandas DataFrame and verify:
- Shape, column names, dtypes
- All unique `statusDescription` values (expected: 7 + null)
- All unique `resultCode` values (expected: 16 + null)
- Null counts per column
- Offshore vs onshore split
- First five rows

Flag anything that deviates from the spec before proceeding to Section 3.

In [ ]:
import pandas as pd

# Expected values from the data spec — used to detect drift
_EXPECTED_STATUSES = {
    "Plugged and abandoned",
    "Producing/Injecting",
    "Monitoring",
    "Suspended",
    "Closed-In",
    "Sidetracked",
    None,  # ghost wells
}

_EXPECTED_RESULT_CODES = {
    "GAS", "OIL", "OAG", "GOS", "GSS", "OLS",  # hydrocarbon — Group A candidates
    "DRY", "SHW", "INC", "UNK", "WTR", "CNS",  # non-hydrocarbon
    "INJ", "STO", "GNS", "OGS",                 # injection / storage
    None,
}

In [ ]:
def load_raw_json(path: Path) -> pd.DataFrame:
    """Load the raw borehole JSON into a DataFrame."""
    data = json.loads(path.read_text())
    df = pd.DataFrame(data)
    logger.info("Loaded DataFrame: %d rows × %d cols", *df.shape)
    return df


def convert_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Convert Unix-millisecond date columns to pandas datetime (UTC)."""
    for col in ("startDate", "endDate", "confidentialityDate"):
        df[col] = pd.to_datetime(df[col], unit="ms", utc=True, errors="coerce")
    return df


def print_shape_and_dtypes(df: pd.DataFrame) -> None:
    """Print the DataFrame shape and column dtypes."""
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    print("Column dtypes:")
    print(df.dtypes.to_string())


def print_unique_values(df: pd.DataFrame, col: str, expected: set) -> None:
    """Print all unique values for a column and flag any that differ from the spec."""
    actual = set(df[col].unique())  # includes NaN as float nan
    # Normalise: pandas represents None/null as np.nan for object cols
    actual_normalised = {None if pd.isna(v) else v for v in actual}
    unexpected = actual_normalised - expected
    missing = expected - actual_normalised

    counts = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="count")
    print(f"\n--- {col} ({len(actual_normalised)} unique values) ---")
    print(counts.to_string(index=False))

    if unexpected:
        logger.warning("UNEXPECTED values in %s: %s — flag before proceeding", col, unexpected)
    if missing:
        logger.info("Values in spec but absent from data (may be normal): %s", missing)
    if not unexpected:
        logger.info("%s values match spec ✓", col)


def print_null_counts(df: pd.DataFrame) -> None:
    """Print null count and percentage for every column."""
    nulls = df.isnull().sum()
    pct = (nulls / len(df) * 100).round(1)
    summary = pd.DataFrame({"null_count": nulls, "null_pct": pct})
    summary = summary[summary["null_count"] > 0].sort_values("null_count", ascending=False)
    print("\n--- Null counts (columns with at least one null) ---")
    print(summary.to_string())


def print_on_offshore_split(df: pd.DataFrame) -> None:
    """Print count of onshore vs offshore records."""
    split = df["onOffshore"].value_counts(dropna=False)
    print("\n--- onOffshore split ---")
    print(split.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 2
# ---------------------------------------------------------------------------

df = load_raw_json(RAW_JSON_PATH)
df = convert_dates(df)

print_shape_and_dtypes(df)
print_unique_values(df, "statusDescription", _EXPECTED_STATUSES)
print_unique_values(df, "resultCode", _EXPECTED_RESULT_CODES)
print_null_counts(df)
print_on_offshore_split(df)

print("\n--- First five rows (key columns) ---")
key_cols = ["boreholeName", "clientOrgName", "statusDescription",
            "resultCode", "onOffshore", "startDate", "endDate", "blockCd"]
print(df[key_cols].head().to_string(index=False))

---
## Section 3 — Filter

Apply offshore filter then split into three groups:

| Group | Criteria | Story role |
|---|---|---|
| **A** | `statusDescription` in Suspended / Closed-In, `onOffshore == OFF` | Core accountability table |
| **B** | `statusDescription == Sidetracked`, `onOffshore == OFF` | Secondary — incomplete wells |
| **Ghost** | `statusDescription` is null, `onOffshore == OFF` | Audit gap — no status on record |

**No `resultCode` filter on Group A** — all inactive offshore wells carry decommissioning
obligations regardless of result. `resultCode` is retained for breakdown in Section 4.

Expected: Group A = 320, Group B = 249, ghost = 44.

In [ ]:
# ---------------------------------------------------------------------------
# Filter constants
# ---------------------------------------------------------------------------

_INACTIVE_STATUSES = {"Suspended", "Closed-In"}

# Hydrocarbon result codes — updated to include OWGS and GWOS
# (OWGS = oil/water/gas/something, GWOS = gas/water/oil/something)
_HC_RESULT_CODES = {"GAS", "OIL", "OAG", "GOS", "GSS", "OLS", "OWGS", "GWOS"}

_EXPECTED_GROUP_A   = 320
_EXPECTED_GROUP_B   = 249
_EXPECTED_GHOST     = 44
_COUNT_TOLERANCE    = 20  # flag if actual count differs by more than this

In [ ]:
def filter_offshore(df: pd.DataFrame) -> pd.DataFrame:
    """Return only offshore wells (onOffshore == 'OFF')."""
    offshore = df[df["onOffshore"] == "OFF"].copy()
    logger.info("Offshore wells: %d (of %d total)", len(offshore), len(df))
    return offshore


def split_group_a(offshore: pd.DataFrame) -> pd.DataFrame:
    """Inactive offshore wells — core accountability table.

    All Suspended and Closed-In offshore wells regardless of resultCode.
    Non-hydrocarbon wells (dry holes, water wells etc.) carry the same
    decommissioning obligation as hydrocarbon wells. resultCode is retained
    as a column for breakdown in Section 4.

    Note: the original spec listed a resultCode filter but its own count
    of 320 was calculated without it. Dropping the filter reaches 320.
    """
    mask = offshore["statusDescription"].isin(_INACTIVE_STATUSES)
    return offshore[mask].copy()


def split_group_b(offshore: pd.DataFrame) -> pd.DataFrame:
    """Sidetracked offshore wells — incomplete / abandoned mid-drill."""
    mask = offshore["statusDescription"] == "Sidetracked"
    return offshore[mask].copy()


def split_ghost(offshore: pd.DataFrame) -> pd.DataFrame:
    """Offshore wells with no status on record — audit gap."""
    mask = offshore["statusDescription"].isna()
    return offshore[mask].copy()


def check_group_count(label: str, df: pd.DataFrame, expected: int) -> None:
    """Warn if group count differs from expected by more than the tolerance."""
    delta = abs(len(df) - expected)
    if delta > _COUNT_TOLERANCE:
        logger.warning(
            "%s count %d differs from expected %d (delta %d) — investigate before proceeding",
            label, len(df), expected, delta,
        )
    else:
        logger.info("%s: %d records (expected ~%d) ✓", label, len(df), expected)


In [ ]:
# ---------------------------------------------------------------------------
# Run Section 3
# ---------------------------------------------------------------------------

offshore = filter_offshore(df)

group_a = split_group_a(offshore)
group_b = split_group_b(offshore)
ghost   = split_ghost(offshore)

check_group_count("Group A", group_a, _EXPECTED_GROUP_A)
check_group_count("Group B", group_b, _EXPECTED_GROUP_B)
check_group_count("Ghost  ", ghost,   _EXPECTED_GHOST)

print("\n--- Offshore wells excluded from all groups ---")
excluded = offshore[
    ~offshore.index.isin(group_a.index)
    & ~offshore.index.isin(group_b.index)
    & ~offshore.index.isin(ghost.index)
]
excl_counts = excluded["statusDescription"].value_counts(dropna=False)
print(excl_counts.to_string())
print(f"\nTotal offshore: {len(offshore)}")
print(f"Group A:        {len(group_a)}")
print(f"Group B:        {len(group_b)}")
print(f"Ghost:          {len(ghost)}")
print(f"Excluded:       {len(excluded)}")
print(f"Check sum:      {len(group_a) + len(group_b) + len(ghost) + len(excluded)} (should equal {len(offshore)})")

print("\n--- Group A sample (first 5 rows) ---")
print(group_a[["boreholeName", "clientOrgName", "statusDescription",
               "resultCode", "startDate", "endDate", "blockCd"]].head().to_string(index=False))

---
## Section 4 — Operator Analysis

Group A wells ranked by operator. For each operator:

- **well_count** — total inactive offshore wells
- **result_codes** — unique result codes across their wells
- **earliest_spud** — earliest `startDate` (oldest well in portfolio)
- **latest_cessation** — most recent `endDate`
- **mean_years_inactive** — mean years since `endDate` (null endDates excluded)
- **ownership_transfers** — count of wells where `clientOrgName` ≠ `legalOwnerName`

Wells where `clientOrgName` differs from `legalOwnerName` are flagged separately —
these have changed hands and the current legal owner may differ from the operator
of record, which is significant for the accountability story.

Saved to `netherlands/data/processed/group_a_by_operator.csv`.

In [ ]:
import numpy as np

TODAY = pd.Timestamp.now(tz="UTC")


def years_since(ts: pd.Series) -> pd.Series:
    """Return fractional years between a datetime Series and today. Nulls propagate."""
    return (TODAY - ts).dt.days / 365.25


def flag_ownership_transfers(group_a: pd.DataFrame) -> pd.DataFrame:
    """Return rows where clientOrgName differs from legalOwnerName.

    These are wells that have changed hands. The current legal owner
    may bear the decommissioning liability even if the operator name differs.
    """
    mask = (
        group_a["clientOrgName"].notna()
        & group_a["legalOwnerName"].notna()
        & (group_a["clientOrgName"] != group_a["legalOwnerName"])
    )
    transfers = group_a[mask][["boreholeName", "clientOrgName",
                                "legalOwnerName", "statusDescription",
                                "resultCode", "endDate"]].copy()
    return transfers


def build_operator_table(group_a: pd.DataFrame) -> pd.DataFrame:
    """Aggregate Group A wells by clientOrgName into a ranked operator table."""
    # Fill null clientOrgName for grouping — these are flagged separately
    working = group_a.copy()
    working["clientOrgName"] = working["clientOrgName"].fillna("[Unknown operator]")

    # Ownership transfer flag per well
    working["is_transfer"] = (
        working["clientOrgName"].notna()
        & working["legalOwnerName"].notna()
        & (working["clientOrgName"] != working["legalOwnerName"])
    ).astype(int)

    # Years inactive per well
    working["years_inactive"] = years_since(working["endDate"])

    table = (
        working.groupby("clientOrgName", dropna=False)
        .agg(
            well_count=("boreholeDbk", "count"),
            result_codes=("resultCode", lambda s: ", ".join(sorted(s.dropna().unique()))),
            earliest_spud=("startDate", "min"),
            latest_cessation=("endDate", "max"),
            mean_years_inactive=("years_inactive", "mean"),
            ownership_transfers=("is_transfer", "sum"),
        )
        .reset_index()
        .sort_values("well_count", ascending=False)
        .reset_index(drop=True)
    )

    # Round years for readability
    table["mean_years_inactive"] = table["mean_years_inactive"].round(1)
    # Format dates as year only for the summary table
    table["earliest_spud"] = table["earliest_spud"].dt.year
    table["latest_cessation"] = table["latest_cessation"].dt.year

    return table

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 4
# ---------------------------------------------------------------------------

transfers = flag_ownership_transfers(group_a)
operator_table = build_operator_table(group_a)

print(f"Group A wells:      {len(group_a)}")
print(f"Unique operators:   {group_a['clientOrgName'].nunique()} "
      f"(+{group_a['clientOrgName'].isna().sum()} with null clientOrgName)")
print(f"Ownership transfers:{len(transfers)} wells where clientOrgName ≠ legalOwnerName")

print("\n--- Operator table (ranked by well count) ---")
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 45)
pd.set_option("display.width", 200)
print(operator_table.to_string(index=False))

print("\n--- Ownership transfer wells ---")
if len(transfers):
    print(transfers.to_string(index=False))
else:
    print("None found.")

# Save
out_path = DATA_PROCESSED / "group_a_by_operator.csv"
operator_table.to_csv(out_path, index=False)
logger.info("Saved operator table (%d rows) to %s", len(operator_table), out_path)

---
## Section 5 — Production Crossref

The NLOG production API returns monthly Gas/Oil figures per well per year.
We use it to identify two subsets of Group A:

1. **Null endDate wells** — `endDate` is missing in the borehole data but production
   figures confirm zero output for 2+ consecutive recent years. These are the most
   significant: no formal cessation record exists but the well is clearly inactive.

2. **Recent cessation wells** — `endDate` is populated but production figures show
   zero output starting earlier than the recorded `endDate`. The formal record lags reality.

**API parameters** (discovered from Network tab):  
`POST /nlog-mapviewer/rest/prodfigures/well`  
`{"yearStart": N, "yearEnd": N, "product": "Gas"|"Oil", "production": "Produced"}`

Data available from 2003 only. We fetch 2020–2024 (5 years) for both Gas and Oil.
The session cookie from Section 1 is reused.

Saved to `netherlands/data/processed/informal_cessation_wells.csv`.

In [ ]:
_PROD_URL = "https://www.nlog.nl/nlog-mapviewer/rest/prodfigures/well"
_PROD_YEARS = list(range(2020, 2025))  # 2020–2024 inclusive
_PRODUCTS = ["Gas", "Oil"]
_CONSECUTIVE_ZERO_THRESHOLD = 2  # years of zero production = informal cessation


def fetch_production_year(session: requests.Session, year: int, product: str) -> list:
    """Fetch monthly production figures for all wells for one year and product."""
    payload = {
        "yearStart": year,
        "yearEnd": year,
        "product": product,
        "production": "Produced",
    }
    resp = session.post(_PROD_URL, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    logger.info("Production %s %d → %d records", product, year, len(data))
    return data


def probe_response_shape(data: list) -> None:
    """Print the first record to understand the response schema."""
    if data:
        print("\n--- Production API response shape (first record) ---")
        print(json.dumps(data[0], indent=2, ensure_ascii=False))
    else:
        print("Empty response — no production records for this year/product.")

In [ ]:
# ---------------------------------------------------------------------------
# Step 1 — probe the response shape with a single request before fetching all
# ---------------------------------------------------------------------------

probe_data = fetch_production_year(session, 2023, "Gas")
probe_response_shape(probe_data)

In [ ]:
# ---------------------------------------------------------------------------
# Step 2 — fetch all years and products, build a tidy production DataFrame
# Run this cell AFTER confirming the response shape above looks correct.
# ---------------------------------------------------------------------------

def build_production_dataframe(session: requests.Session,
                               years: list, products: list) -> pd.DataFrame:
    """Fetch production for all year/product combinations and return a tidy DataFrame.

    Columns in the output depend on the API response shape confirmed in the probe.
    After fetching, we normalise to at minimum:
      boreholeName, year, product, total_production
    """
    frames = []
    for product in products:
        for year in years:
            data = fetch_production_year(session, year, product)
            if not data:
                continue
            df_year = pd.DataFrame(data)
            df_year["_year"] = year
            df_year["_product"] = product
            frames.append(df_year)
    if not frames:
        logger.warning("No production data returned for any year/product combination.")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


prod_raw = build_production_dataframe(session, _PROD_YEARS, _PRODUCTS)
print(f"\nProduction DataFrame shape: {prod_raw.shape}")
print("Columns:", list(prod_raw.columns))
print("\nFirst 3 rows:")
print(prod_raw.head(3).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Step 3 — aggregate to annual total per well, then crossref with Group A
# NOTE: update 'name_col' and 'value_cols' below to match the actual column
# names confirmed by the probe above.
# ---------------------------------------------------------------------------

def summarise_annual_production(prod_raw: pd.DataFrame) -> pd.DataFrame:
    """Aggregate monthly production records to annual totals per well.

    Returns a DataFrame with columns: boreholeName, year, product, annual_total.
    Adapt the aggregation columns to match the actual API response schema.
    """
    # Identify the well name column and numeric production columns
    # These are confirmed from the probe — update if the API uses different names
    possible_name_cols = ["boreholeName", "wellName", "name", "boreholeNm", "wellNm"]
    name_col = next((c for c in possible_name_cols if c in prod_raw.columns), None)
    if name_col is None:
        raise ValueError(
            f"Cannot find well name column. Available columns: {list(prod_raw.columns)}. "
            "Update possible_name_cols to match the actual API response."
        )

    # Sum all numeric columns except year (proxy for monthly totals)
    numeric_cols = prod_raw.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ("_year",)]

    prod_raw["annual_total"] = prod_raw[numeric_cols].sum(axis=1)

    annual = (
        prod_raw.groupby([name_col, "_year", "_product"])["annual_total"]
        .sum()
        .reset_index()
        .rename(columns={name_col: "boreholeName", "_year": "year", "_product": "product"})
    )
    return annual


def find_consecutive_zero_years(annual: pd.DataFrame,
                                threshold: int) -> pd.DataFrame:
    """Return wells with zero production for `threshold` or more consecutive years.

    A well 'year' is zero if it either appears in the data with zero production
    OR does not appear at all (absent = no production reported).
    """
    # Build a pivot: wells × years, fill missing with 0
    pivot = (
        annual.groupby(["boreholeName", "year"])["annual_total"]
        .sum()
        .unstack(fill_value=0)
    )
    years_sorted = sorted(pivot.columns)
    pivot = pivot[years_sorted]

    # Find wells with >= threshold consecutive zero years
    results = []
    for well, row in pivot.iterrows():
        consecutive = 0
        max_consecutive = 0
        last_zero_start = None
        for yr in years_sorted:
            if row[yr] == 0:
                if consecutive == 0:
                    last_zero_start = yr
                consecutive += 1
                max_consecutive = max(max_consecutive, consecutive)
            else:
                consecutive = 0
        if max_consecutive >= threshold:
            results.append({
                "boreholeName": well,
                "consecutive_zero_years": max_consecutive,
                "zero_from_year": last_zero_start,
            })
    return pd.DataFrame(results)


def crossref_with_group_a(group_a: pd.DataFrame,
                          zero_wells: pd.DataFrame,
                          annual: pd.DataFrame) -> pd.DataFrame:
    """Join zero-production wells back to Group A to produce the informal cessation table."""
    ga_cols = ["boreholeName", "clientOrgName", "legalOwnerName",
               "statusDescription", "resultCode", "endDate", "blockCd"]
    ga_slim = group_a[ga_cols].copy()

    # Flag null endDate wells
    ga_slim["null_end_date"] = ga_slim["endDate"].isna()

    # Join zero-production findings
    merged = ga_slim.merge(zero_wells, on="boreholeName", how="inner")
    merged = merged.sort_values(
        ["consecutive_zero_years", "null_end_date"], ascending=[False, False]
    ).reset_index(drop=True)
    return merged

In [ ]:
# ---------------------------------------------------------------------------
# Run Steps 3–5
# ---------------------------------------------------------------------------

annual = summarise_annual_production(prod_raw)
print(f"Annual production summary: {len(annual)} well-year-product rows")
print(f"Unique wells with any production 2020-2024: {annual['boreholeName'].nunique()}")

zero_wells = find_consecutive_zero_years(annual, _CONSECUTIVE_ZERO_THRESHOLD)
print(f"\nWells with {_CONSECUTIVE_ZERO_THRESHOLD}+ consecutive zero-production years: {len(zero_wells)}")

# Group A wells with null endDate
null_end = group_a[group_a["endDate"].isna()]
print(f"Group A wells with null endDate: {len(null_end)}")
if len(null_end):
    print(null_end[["boreholeName", "clientOrgName", "statusDescription", "resultCode"]].to_string(index=False))

informal = crossref_with_group_a(group_a, zero_wells, annual)
print(f"\nInformal cessation candidates (Group A × zero production): {len(informal)}")
print(informal.to_string(index=False))

# Save
out_path = DATA_PROCESSED / "informal_cessation_wells.csv"
informal.to_csv(out_path, index=False)
logger.info("Saved %d informal cessation wells to %s", len(informal), out_path)

---
## Section 6 — Join and Summarise

Combine the Group A operator table (Section 4) with the production crossref
(Section 5) to produce the final accountability table.

**Per legal owner** (the entity that actually holds the decommissioning liability):
- Total inactive well count
- Mean years inactive
- Count of wells confirmed inactive by production data (`confirmed_inactive`)
- Count of wells where `clientOrgName` ≠ `legalOwnerName` (transferred wells)
- Ghost well count

**Per operator** (clientOrgName — who drilled / last operated):
- Same metrics, for cross-reference

The legal owner table is the primary accountability table — EBN holds a statutory
share in all Dutch licences, so the legal owner is the entity on the hook.

Saved to `netherlands/outputs/nl_wells_analysis.csv`.

In [ ]:
def build_accountability_table(group_a: pd.DataFrame,
                               informal: pd.DataFrame,
                               ghost: pd.DataFrame,
                               group_by_col: str) -> pd.DataFrame:
    """Build the final accountability table grouped by legal owner or operator.

    Args:
        group_a:    Group A DataFrame (320 inactive offshore wells)
        informal:   Informal cessation candidates from Section 5
        ghost:      Ghost wells (null status offshore)
        group_by_col: 'legalOwnerName' or 'clientOrgName'
    """
    working = group_a.copy()
    working[group_by_col] = working[group_by_col].fillna("[Unknown]")

    # Flag confirmed-inactive wells (appear in production crossref)
    confirmed_wells = set(informal["boreholeName"])
    working["confirmed_inactive"] = working["boreholeName"].isin(confirmed_wells).astype(int)

    # Flag ownership transfers
    working["is_transfer"] = (
        working["clientOrgName"].notna()
        & working["legalOwnerName"].notna()
        & (working["clientOrgName"] != working["legalOwnerName"])
    ).astype(int)

    # Years inactive
    working["years_inactive"] = years_since(working["endDate"])

    # Ghost wells per legal owner
    ghost_working = ghost.copy()
    ghost_working[group_by_col] = ghost_working[group_by_col].fillna("[Unknown]")
    ghost_counts = ghost_working.groupby(group_by_col).size().rename("ghost_wells")

    table = (
        working.groupby(group_by_col, dropna=False)
        .agg(
            inactive_well_count=("boreholeDbk", "count"),
            mean_years_inactive=("years_inactive", "mean"),
            max_years_inactive=("years_inactive", "max"),
            confirmed_inactive=("confirmed_inactive", "sum"),
            transferred_wells=("is_transfer", "sum"),
            result_codes=("resultCode", lambda s: ", ".join(sorted(s.dropna().unique()))),
            earliest_spud=("startDate", "min"),
        )
        .reset_index()
        .join(ghost_counts, on=group_by_col, how="left")
    )

    table["ghost_wells"] = table["ghost_wells"].fillna(0).astype(int)
    table["mean_years_inactive"] = table["mean_years_inactive"].round(1)
    table["max_years_inactive"] = table["max_years_inactive"].round(1)
    table["earliest_spud"] = table["earliest_spud"].dt.year

    return table.sort_values("inactive_well_count", ascending=False).reset_index(drop=True)

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 6
# ---------------------------------------------------------------------------

# Primary table — by legal owner (holds the decommissioning liability)
by_legal_owner = build_accountability_table(group_a, informal, ghost, "legalOwnerName")

# Secondary table — by operator (who drilled / last ran the well)
by_operator = build_accountability_table(group_a, informal, ghost, "clientOrgName")

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 220)

print("=" * 80)
print("ACCOUNTABILITY TABLE — BY LEGAL OWNER (decommissioning liability holder)")
print("=" * 80)
print(by_legal_owner.to_string(index=False))

print("\n" + "=" * 80)
print("REFERENCE TABLE — BY OPERATOR (clientOrgName)")
print("=" * 80)
print(by_operator.to_string(index=False))

# Save final output
out_path = OUTPUTS_DIR / "nl_wells_analysis.csv"
by_legal_owner.to_csv(out_path, index=False)
logger.info("Saved accountability table (%d legal owners) to %s",
            len(by_legal_owner), out_path)

# Also save operator view alongside
op_path = OUTPUTS_DIR / "nl_wells_by_operator.csv"
by_operator.to_csv(op_path, index=False)
logger.info("Saved operator table (%d operators) to %s", len(by_operator), op_path)

---
## Section 7 — Sense Check and Outputs

Final verification against the data diary before signing off the notebook.

**Expected totals (data diary):**
| Metric | Expected |
|---|---|
| Offshore total | 2,195 |
| Group A | 320 |
| Group B | 249 |
| Ghost wells | 44 |

Then: top 5 operators by inactive well count, and full ghost well listing.

In [ ]:
# ---------------------------------------------------------------------------
# Data diary totals
# ---------------------------------------------------------------------------

_DIARY = {
    "offshore_total": 2195,
    "group_a": 320,
    "group_b": 249,
    "ghost": 44,
}


def verify_totals(offshore: pd.DataFrame,
                  group_a: pd.DataFrame,
                  group_b: pd.DataFrame,
                  ghost: pd.DataFrame,
                  diary: dict) -> None:
    """Compare pipeline totals against data diary entries and flag any discrepancy."""
    actuals = {
        "offshore_total": len(offshore),
        "group_a":        len(group_a),
        "group_b":        len(group_b),
        "ghost":          len(ghost),
    }
    all_ok = True
    print(f"{'Metric':<20} {'Expected':>10} {'Actual':>10} {'Status':>8}")
    print("-" * 52)
    for key, expected in diary.items():
        actual = actuals[key]
        status = "✓" if actual == expected else "⚠ MISMATCH"
        if actual != expected:
            all_ok = False
        print(f"{key:<20} {expected:>10,} {actual:>10,} {status:>8}")
    print()
    if all_ok:
        logger.info("All totals match data diary ✓")
    else:
        logger.warning("One or more totals do not match — investigate before publishing.")

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 7
# ---------------------------------------------------------------------------

print("--- Data diary verification ---")
verify_totals(offshore, group_a, group_b, ghost, _DIARY)

print("--- Top 5 legal owners by inactive well count ---")
print(by_legal_owner.head(5)[
    ["legalOwnerName", "inactive_well_count", "mean_years_inactive",
     "confirmed_inactive", "transferred_wells", "ghost_wells"]
].to_string(index=False))

print("\n--- Top 5 operators by inactive well count ---")
print(by_operator.head(5)[
    ["clientOrgName", "inactive_well_count", "mean_years_inactive",
     "confirmed_inactive", "transferred_wells"]
].to_string(index=False))

print("\n--- All ghost wells (null status, offshore) ---")
ghost_display = ghost[
    ["boreholeName", "clientOrgName", "legalOwnerName", "startDate", "blockCd"]
].copy()
ghost_display["startDate"] = ghost_display["startDate"].dt.year
ghost_display = ghost_display.sort_values("startDate")
print(ghost_display.to_string(index=False))
print(f"\nTotal ghost wells: {len(ghost)}")

print("\n--- Notable findings for data diary ---")
print(f"Tenneco Netherlands Inc.: {by_legal_owner[by_legal_owner['legalOwnerName'] == 'Tenneco Netherlands Inc.']['inactive_well_count'].values[0]} wells, "
      f"mean {by_legal_owner[by_legal_owner['legalOwnerName'] == 'Tenneco Netherlands Inc.']['mean_years_inactive'].values[0]} yrs inactive, "
      f"0 transfers — operator may be dormant")
print(f"Wells with 100% transfer rate (current legal owner ≠ original operator):")
full_transfer = by_legal_owner[by_legal_owner['inactive_well_count'] == by_legal_owner['transferred_wells']]
for _, row in full_transfer.iterrows():
    print(f"  {row['legalOwnerName']}: {row['inactive_well_count']} wells")
print(f"\nTotal Group A wells transferred to a different legal owner: "
      f"{group_a[group_a['clientOrgName'] != group_a['legalOwnerName']].shape[0]} / {len(group_a)} "
      f"({group_a[group_a['clientOrgName'] != group_a['legalOwnerName']].shape[0]/len(group_a)*100:.0f}%)")